<a href="https://colab.research.google.com/github/darshanlahamage/Adaptive-Adversarial-Augmentation/blob/main/vcg_adersarial_aug2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --quiet torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 || true
!pip install --quiet scikit-learn matplotlib tqdm

In [ ]:
import os, math, time, random, json
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models, utils as vutils
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

def get_device(prefer_cuda=True):
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

device = get_device()
print("Device:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

def accuracy_from_logits(logits, labels):
    preds = logits.argmax(dim=1)
    return (preds == labels).float().mean().item()

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

Device: cuda


In [ ]:
# Cell 3: calibration (ECE) and Failure Diversity Score (FDS)

def compute_ece(probs, correct, n_bins=15):
    """
    probs: numpy array of predicted confidences (max-prob) per example
    correct: numpy boolean array (1 if correct)
    returns scalar ECE
    """
    bin_edges = np.linspace(0.0, 1.0, n_bins+1)
    ece = 0.0
    n = probs.shape[0]
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i+1]
        mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        acc = correct[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)

def compute_failure_diversity(features, pred_labels, true_labels, n_clusters=10, pca_dim=64, random_state=0):
    """
    - features: numpy array (N, D) of penultimate features for all examples
    - pred_labels, true_labels: numpy arrays
    Returns normalized entropy (0..1) and failure_count
    """
    mask = pred_labels != true_labels
    if mask.sum() == 0:
        return 0.0, 0  # no failures
    X = features[mask]
    n_samples = X.shape[0]
    pca_dim = min(pca_dim, X.shape[1])
    if n_samples <= 2:
        return 0.0, int(n_samples)
    pca = PCA(n_components=pca_dim, random_state=random_state)
    Xr = pca.fit_transform(X)
    k = min(n_clusters, max(2, n_samples // 10))
    kmeans = KMeans(n_clusters=k, random_state=random_state).fit(Xr)
    clusters = kmeans.labels_
    counts = np.bincount(clusters, minlength=k)
    probs = counts / counts.sum()
    ent = -np.sum([p * math.log(p + 1e-12) for p in probs])
    max_ent = math.log(k)
    normalized_ent = ent / (max_ent + 1e-12)
    return float(normalized_ent), int(n_samples)

In [ ]:
# Cell 4: model wrapper that exposes penultimate features

class ResNetFeatureWrapper(nn.Module):
    """
    Wraps torchvision.resnet18, adapts final FC for num_classes,
    and exposes extract_features(x) that returns penultimate features (flattened avgpool).
    """
    def __init__(self, num_classes=10, pretrained=False):
        super().__init__()
        self.model = models.resnet18(pretrained=pretrained)
        in_feats = self.model.fc.in_features
        self.model.fc = nn.Linear(in_feats, num_classes)

    def forward(self, x):
        return self.model(x)

    def extract_features(self, x):
        # replicate forward up to avgpool -> flatten
        x = self.model.conv1(x)
        x = self.model.bn1(x)
        x = self.model.relu(x)
        x = self.model.maxpool(x)

        x = self.model.layer1(x)
        x = self.model.layer2(x)
        x = self.model.layer3(x)
        x = self.model.layer4(x)

        x = self.model.avgpool(x)
        feat = torch.flatten(x, 1)
        return feat

In [ ]:
class TransformGenerator(nn.Module):

    def __init__(self, hidden=64):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, hidden, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden, hidden, 3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(hidden, 7)

    def forward(self, x):

        z = self.encoder(x)
        z = torch.flatten(z,1)

        params = self.fc(z)

        rot = torch.tanh(params[:,0]) * 20
        tx  = torch.tanh(params[:,1]) * 4
        ty  = torch.tanh(params[:,2]) * 4

        brightness = torch.tanh(params[:,3]) * 0.2
        contrast   = 1 + torch.tanh(params[:,4]) * 0.2

        blur_sigma = torch.relu(params[:,5])
        cutout     = torch.sigmoid(params[:,6]) * 0.4

        return rot,tx,ty,brightness,contrast,blur_sigma,cutout

In [ ]:
def spatial_transform(x, rot, tx, ty):

    B,C,H,W = x.shape

    theta = torch.zeros(B,2,3).to(x.device)

    angle = rot * math.pi / 180

    theta[:,0,0] = torch.cos(angle)
    theta[:,0,1] = -torch.sin(angle)

    theta[:,1,0] = torch.sin(angle)
    theta[:,1,1] = torch.cos(angle)

    theta[:,0,2] = tx / W
    theta[:,1,2] = ty / H

    grid = F.affine_grid(theta, x.size(), align_corners=False)

    return F.grid_sample(x, grid, align_corners=False)

In [ ]:
def photometric_transform(x, brightness, contrast):

    x = x * contrast.view(-1,1,1,1)
    x = x + brightness.view(-1,1,1,1)

    return torch.clamp(x,0,1)

In [ ]:
import torchvision
def blur_transform(x, sigma):

    if sigma.mean() < 0.05:
        return x

    return torchvision.transforms.functional.gaussian_blur(x, 5)

In [ ]:
def cutout_transform(x, strength):

    B,C,H,W = x.shape
    mask = torch.ones_like(x)

    size = (strength * H).long()

    for i in range(B):

        s = int(size[i].item())
        if s <= 1:
            continue

        cx = torch.randint(0,W,(1,))
        cy = torch.randint(0,H,(1,))

        x1 = max(cx - s//2,0)
        x2 = min(cx + s//2,W)

        y1 = max(cy - s//2,0)
        y2 = min(cy + s//2,H)

        mask[i,:,y1:y2,x1:x2] = 0

    return x * mask

In [ ]:
def apply_generator(x, params):

    rot,tx,ty,b,c,blur_sigma,cutout = params

    x = spatial_transform(x,rot,tx,ty)

    x = photometric_transform(x,b,c)

    x = blur_transform(x,blur_sigma)

    x = cutout_transform(x,cutout)

    return x

In [ ]:
def generator_regularizer(params, feat_orig, feat_aug,
                          alpha_param=0.01, alpha_feat=1.0):

    reg_param = 0
    for p in params:
        reg_param += torch.mean(p**2)

    reg_feat = torch.mean((feat_aug - feat_orig)**2)

    return alpha_param * reg_param + alpha_feat * reg_feat

In [ ]:
# Cell 6: datasets and dataloaders

def get_cifar10_loaders(data_dir="./data", batch_size=128, num_workers=4):
    cifar_mean = (0.4914, 0.4822, 0.4465)
    cifar_std  = (0.2470, 0.2435, 0.2616)

    train_transforms = [
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip()
    ]
    train_transforms += [transforms.ToTensor(), transforms.Normalize(cifar_mean, cifar_std)]

    test_transforms = transforms.Compose([transforms.ToTensor(), transforms.Normalize(cifar_mean, cifar_std)])

    train_tf = transforms.Compose(train_transforms)

    train_ds = datasets.CIFAR10(root=data_dir, train=True, download=True, transform=train_tf)
    test_ds  = datasets.CIFAR10(root=data_dir, train=False, download=True, transform=test_transforms)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    return train_loader, test_loader

In [ ]:
# Cell 7: functions to unnormalize images and save visualization grid of augmented images

cifar_mean = np.array([0.4914, 0.4822, 0.4465])
cifar_std  = np.array([0.2470, 0.2435, 0.2616])

def unnormalize_tensor(img_tensor):
    """ img_tensor: Tensor [C,H,W] normalized; returns numpy HxWxC in [0,1] """
    arr = img_tensor.cpu().numpy().transpose(1,2,0)
    arr = (arr * cifar_std) + cifar_mean
    arr = np.clip(arr, 0.0, 1.0)
    return arr

def save_image_grid(tensor_list, filename, nrow=8, pad_value=0.03):
    """
    tensor_list: list of tensors [C,H,W] in normalized space (same normalization as model's input)
    Save a grid (unnormalized for visualization).
    """
    # stack and make grid with torchvision
    norm_stack = torch.stack(tensor_list, dim=0)  # [N,C,H,W]
    # convert normalized to original [0,1]
    # first unnormalize via tensor ops for efficiency
    mean = torch.tensor(cifar_mean).view(1,3,1,1).to(norm_stack.device)
    std  = torch.tensor(cifar_std).view(1,3,1,1).to(norm_stack.device)
    unnorm = norm_stack * std + mean
    unnorm = torch.clamp(unnorm, 0.0, 1.0)
    grid = vutils.make_grid(unnorm, nrow=nrow, padding=2, pad_value=pad_value)
    vutils.save_image(grid, filename)

In [ ]:
# Cell 8: safe, explicit minimax update functions (single-batch operations)
# This maps to the math and autograd patterns we discussed.

def classifier_k_steps(images_norm, labels, model, gen, optimizer_clf, criterion, k_steps=3, device='cpu'):
    """
    Perform k classifier updates on a single batch.
    images_norm: normalized images tensor [B,C,H,W] (ready for model forward)
    gen: generator (used to create augmentations), but we use torch.no_grad() for generator forward
    Returns last loss scalar.
    """
    model.train()
    gen.eval()  # generator in eval when used for classifier steps (we don't want to update it)
    last_loss = None
    for _ in range(k_steps):
        optimizer_clf.zero_grad()
        with torch.no_grad():
            # run generator in no_grad to avoid building graph for phi
            # But generator expects unnormalized images in [0,1], so unnormalize model input first.
            mean = torch.tensor(cifar_mean, dtype=torch.float).view(1,3,1,1).to(images_norm.device)
            std  = torch.tensor(cifar_std, dtype=torch.float).view(1,3,1,1).to(images_norm.device)
            images_01 = images_norm * std + mean      # back to [0,1]
            params = gen(images_01)
            images_aug_01 = apply_generator(images_01, params)
            # re-normalize for model forward
            images_aug = (images_aug_01 - mean) / std
        logits = model(images_aug)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer_clf.step()
        last_loss = float(loss.item())
    return last_loss

def generator_update_step(images_norm, labels, model, gen, optimizer_gen, criterion, lambda_reg=5.0, alpha_l2=1.0, alpha_feat=1.0, device='cpu'):
    """
    One generator update:
      - compute feats_orig (detached) BEFORE calling this function (we will supply it)
      - freeze model params (requires_grad=False) to avoid accumulation of theta grads
      - compute loss_aug and reg, then backward into gen and step
    Returns dict with loss_aug, reg_pix, reg_feat
    """
    # convert normalized images to [0,1] for generator
    mean = torch.tensor(cifar_mean, dtype=torch.float).view(1,3,1,1).to(images_norm.device)
    std  = torch.tensor(cifar_std, dtype=torch.float).view(1,3,1,1).to(images_norm.device)
    images_01 = images_norm * std + mean      # back to [0,1]

    # Freeze classifier params to avoid storing grads for them
    for p in model.parameters():
        p.requires_grad = False

    optimizer_gen.zero_grad()
    params = gen(images_01)                # graph created for gen params
    images_aug_01 = apply_generator(images_01, params)
    # re-normalize for model
    images_aug = (images_aug_01 - mean) / std

    # forward through model (we need gradient w.r.t. inputs so do NOT use no_grad here)
    logits_aug = model(images_aug)
    loss_aug = criterion(logits_aug, labels)      # scalar

    # features - note feats_orig must be computed and passed detached by caller
    feats_aug = model.extract_features(images_aug)

    # generator regularizer requires feats_orig provided externally
    # we'll compute reg externally in main loop where feats_orig is available
    # For modularity this function returns the tensors and scalar loss components.
    # We don't step optimizer_gen here; caller should compute reg, gen_loss and step.
    for p in model.parameters():
        p.requires_grad = True

    # Return useful tensors (delta, images_aug_01, logits_aug, feats_aug, loss_aug)
    return {
        "images_aug_01": images_aug_01.detach(),  # detach copy for possible gallery saving
        "logits_aug": logits_aug,
        "feats_aug": feats_aug,
        "loss_aug": float(loss_aug.item())
    }

In [ ]:
from heapq import nlargest

def minimax_train_epoch(train_loader, model, gen, optim_clf, optim_gen, criterion,
                        epoch, device, save_dir, k_classifier_steps=3,
                        lambda_reg=5.0, alpha_l2=1.0, alpha_feat=1.0,
                        topk_save=8, replay_buffer=None):

    model.to(device)
    gen.to(device)

    model.train()
    gen.train()

    ensure_dir(save_dir)

    stats = {"loss_clf": [], "loss_aug": [], "reg_feat": []}

    gallery_items = []

    mean = torch.tensor(cifar_mean, dtype=torch.float).view(1,3,1,1).to(device)
    std  = torch.tensor(cifar_std, dtype=torch.float).view(1,3,1,1).to(device)

    pbar = tqdm(train_loader, desc=f"Epoch {epoch} train", leave=False)

    for batch_idx, (images, labels) in enumerate(pbar):

        images = images.to(device)
        labels = labels.to(device)

        # ---------------------------------------------------
        # Extract base features (anchor)
        # ---------------------------------------------------

        model.eval()

        with torch.no_grad():
            feats_orig = model.extract_features(images).detach()

        model.train()

        # ---------------------------------------------------
        # Classifier updates
        # ---------------------------------------------------

        loss_clf_val = classifier_k_steps(
            images,
            labels,
            model,
            gen,
            optim_clf,
            criterion,
            k_steps=k_classifier_steps,
            device=device
        )

        stats["loss_clf"].append(loss_clf_val)

        # ---------------------------------------------------
        # Generator update
        # ---------------------------------------------------

        optim_gen.zero_grad()

        images_01 = images * std + mean

        params = gen(images_01)

        images_aug_01 = apply_generator(images_01, params)

        images_aug = (images_aug_01 - mean) / std

        logits_aug = model(images_aug)

        loss_aug = criterion(logits_aug, labels)

        feats_aug = model.extract_features(images_aug)

        reg_feat = torch.mean((feats_aug - feats_orig) ** 2)

        gen_loss = -loss_aug + lambda_reg * reg_feat

        gen_loss.backward()

        torch.nn.utils.clip_grad_norm_(gen.parameters(), 1.0)

        optim_gen.step()

        # ---------------------------------------------------
        # Logging
        # ---------------------------------------------------

        loss_aug_val = float(loss_aug.item())

        stats["loss_aug"].append(loss_aug_val)
        stats["reg_feat"].append(float(reg_feat.item()))

        # ---------------------------------------------------
        # Save augmentations for gallery
        # ---------------------------------------------------

        images_aug_norm = (images_aug_01 - mean) / std

        for i in range(min(topk_save, images_aug_norm.size(0))):

            gallery_items.append(
                (loss_aug_val, images_aug_norm[i].detach().cpu())
            )

        if replay_buffer is not None:

            replay_buffer.append(
                (images_aug_norm.detach().cpu(), labels.detach().cpu())
            )

        pbar.set_postfix({
            "clf_loss": f"{loss_clf_val:.4f}",
            "gen_loss": f"{loss_aug_val:.4f}",
            "feat_reg": f"{reg_feat.item():.5f}"
        })

    gallery_items_sorted = sorted(gallery_items, key=lambda x: -x[0])[:64]

    if len(gallery_items_sorted) > 0:

        tensors = [t for (_, t) in gallery_items_sorted]

        save_path = os.path.join(save_dir, f"gallery_epoch{epoch}.png")

        save_image_grid(tensors, save_path, nrow=8)

    agg = {k: float(np.mean(v)) if len(v) > 0 else 0.0 for k, v in stats.items()}

    return agg

In [ ]:
# Cell 10: evaluation on test loader (computes accuracy, ECE, and optionally FDS)
def evaluate_model(model, loader, device, return_features=False):
    model.to(device)
    model.eval()
    total = 0
    correct = 0
    confs = []
    correct_flags = []

    all_feats = []
    all_preds = []
    all_trues = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            probs = F.softmax(logits, dim=1)
            conf, pred = probs.max(dim=1)
            correct_batch = pred.eq(labels)
            total += images.size(0)
            correct += correct_batch.sum().item()
            confs.append(conf.cpu().numpy())
            correct_flags.append(correct_batch.cpu().numpy())

            if return_features:
                feats = model.extract_features(images).cpu().numpy()
                all_feats.append(feats)
                all_preds.append(pred.cpu().numpy())
                all_trues.append(labels.cpu().numpy())

    acc = correct / total
    confs = np.concatenate(confs)
    correct_flags = np.concatenate(correct_flags)
    ece = compute_ece(confs, correct_flags)
    if return_features:
        feats = np.concatenate(all_feats, axis=0)
        pred_arr = np.concatenate(all_preds, axis=0)
        true_arr = np.concatenate(all_trues, axis=0)
        return acc, ece, feats, pred_arr, true_arr
    return acc, ece

In [ ]:
def run_smoke_test(epochs=2, batch_size=128, use_randaugment=False, k_classifier_steps=3,
                   gen_eps=0.08, gen_lr=1e-4, clf_lr=0.1, lambda_reg=5.0,
                   alpha_l2=1.0, alpha_feat=1.0, save_dir="./runs/run1", device=device):
    ensure_dir(save_dir)
    train_loader, test_loader = get_cifar10_loaders("./data", batch_size=batch_size)
    model = ResNetFeatureWrapper(num_classes=10).to(device)
    gen = TransformGenerator().to(device)

    # optimizers
    optimizer_clf = optim.SGD(model.parameters(), lr=clf_lr, momentum=0.9, weight_decay=5e-4)
    optimizer_gen = optim.Adam(gen.parameters(), lr=gen_lr)

    criterion = nn.CrossEntropyLoss()

    # optional small replay buffer - not used in smoke but available
    replay_buffer = deque(maxlen=200)

    history = defaultdict(list)
    for epoch in range(1, epochs+1):
        t0 = time.time()
        agg = minimax_train_epoch(train_loader, model, gen, optimizer_clf, optimizer_gen, criterion,
                                  epoch, device, save_dir,
                                  k_classifier_steps=k_classifier_steps,
                                  lambda_reg=lambda_reg, alpha_l2=alpha_l2, alpha_feat=alpha_feat,
                                  topk_save=8, replay_buffer=replay_buffer)
        t1 = time.time()
        # eval
        acc, ece = evaluate_model(model, test_loader, device, return_features=False)
        history['train_loss'].append(agg['loss_clf'])
        history['train_loss_aug'].append(agg['loss_aug'])
        history['reg_feat'].append(agg['reg_feat'])
        history['test_acc'].append(acc)
        history['test_ece'].append(ece)
        print(f"Epoch {epoch} time {t1-t0:.1f}s  clf_loss {agg['loss_clf']:.4f}  gen_loss(avg) {agg['loss_aug']:.4f}  reg_feat {agg['reg_feat']:.5f}  test_acc {acc:.4f}  test_ece {ece:.4f}")

    # final detailed eval with features for FDS
    acc, ece, feats, pred_arr, true_arr = evaluate_model(model, test_loader, device, return_features=True)
    fds, n_fail = compute_failure_diversity(feats, pred_arr, true_arr, n_clusters=10)
    print("Final test acc:", acc, "ECE:", ece, "Failure count:", n_fail, "FDS:", fds)

    # save small artifacts
    torch.save({'model_state': model.state_dict(), 'gen_state': gen.state_dict()}, os.path.join(save_dir, 'final_ckpt.pth'))
    with open(os.path.join(save_dir, 'history.json'), 'w') as f:
        json.dump({k: list(v) for k,v in history.items()}, f)
    print("Saved artifacts to", save_dir)
    return model, gen, history

model, gen, history = run_smoke_test(epochs=30, batch_size=64, k_classifier_steps=3, gen_eps=0.06, gen_lr=1e-4, clf_lr=0.05, lambda_reg=5.0, save_dir="./runs_v2/adaptive_aug", device=device)


100%|██████████| 170M/170M [00:05<00:00, 30.3MB/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passin

Epoch 1 time 142.5s  clf_loss 1.7834  gen_loss(avg) 1.7060  reg_feat 11.94710  test_acc 0.4112  test_ece 0.0888


Epoch 2 time 143.8s  clf_loss 1.5005  gen_loss(avg) 1.4263  reg_feat 0.00942  test_acc 0.4462  test_ece 0.0439


Epoch 3 time 143.0s  clf_loss 1.3738  gen_loss(avg) 1.2796  reg_feat 0.01185  test_acc 0.4078  test_ece 0.0521


Epoch 4 time 143.0s  clf_loss 1.3857  gen_loss(avg) 1.2850  reg_feat 0.01021  test_acc 0.4059  test_ece 0.1143


...

Epoch 23 time 142.3s  clf_loss 1.1393  gen_loss(avg) 1.0092  reg_feat 0.01184  test_acc 0.5388  test_ece 0.0793


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch 24 time 142.0s  clf_loss 1.1445  gen_loss(avg) 1.0109  reg_feat 0.01172  test_acc 0.4812  test_ece 0.1600


Epoch 25 time 141.8s  clf_loss 1.0957  gen_loss(avg) 0.9613  reg_feat 0.01296  test_acc 0.5169  test_ece 0.1004


KeyboardInterrupt: 